<a href="https://colab.research.google.com/github/MelisaYasak/transformer_arch/blob/main/sherlockGPT_biagram_to_transformer_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%cd /content/drive/MyDrive/transformers

/content/drive/MyDrive/transformers


# Biagram'dan Self Attention'a

### Hatırlatma! Biagram Kısıtı
Bu örnekte the ve she yalnızca Bigram'ın sınırlılığını göstermek için seçildi. İki bağlamın mutlaka farklı sonraki karakter dağılımlarına sahip olması gerektiğini iddia etmiyoruz.

Bu deneyin öğrettiği: Bigram, önceki karakterlerden yararlanarak tahminini değiştiremiyor.

In [ ]:
#Hücreyi çalıştırmak için sherlockGPT_biagram_language_model.ipynb dosyasındaki BigramLanguageModel classının tanımlı olması gerekmektedir.
import torch
import torch.nn.functional as F

model.eval()

with torch.no_grad():
    x1 = torch.tensor([encode("the")])
    x2 = torch.tensor([encode("she")])

    logits1, _ = model(x1)
    logits2, _ = model(x2)

    p1 = F.softmax(logits1[0, -1], dim=-1)
    p2 = F.softmax(logits2[0, -1], dim=-1)

    print("Aynı dağılım mı?", torch.allclose(p1, p2))
    print("En büyük fark:", (p1 - p2).abs().max().item())

Aynı dağılım mı? True
En büyük fark: 0.0


## Transformer mantığı

Burada mimari bir token için temsil oluştururken önceki tokenlardan bilgi toplayabiliyor. Bunu causal self-attention ile yapıyor.

Bir önceki örnekteki 'the' kelimesini ele alırsak; eğer e yi tahin etmesi gerekirse 't' ve 'h' karakterlerinin bilgisini de kullanacaktı.


### Embedding

Biagramda embeddingleri her karakter için doğrudan tüm olası sonraki karakterlerin logitlerini üretmek için kullandık.

Transformerda ise embedding vektörü öğrenilebilir temsil (ağırlık) olacak.

In [ ]:
import torch
import torch.nn as nn


In [ ]:
torch.manual_seed(42)

n_embed = 4

In [ ]:
token_embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=n_embed
)

In [ ]:
x1 = torch.tensor([encode("the")])
x2 = torch.tensor([encode("she")])

In [ ]:
emb1 = token_embedding(x1)
emb2 = token_embedding(x2)

In [ ]:
print("Girdi boyutu (Input shape): ", x1.shape)
print("Embedding boyutu (Embedding shape): ", emb1.shape)

Girdi boyutu (Input shape):  torch.Size([1, 3])
Embedding boyutu (Embedding shape):  torch.Size([1, 3, 4])


Embedding boyutu (Embedding shape):  torch.Size([1, 3, 4])

Shape'i açıklamak gerekirse:


(batch içinde bir metin örneği var, üç karakter var, her karakter dört boyutlu bir embedding vektörü ile temsil ediliyor)

In [ ]:
print("the - son token embedding: ", emb1[0,-1])
print("she - son token embedding: ", emb2[0,-1])

the - son token embedding:  tensor([ 0.1543,  0.4408, -0.1483, -2.3184], grad_fn=<SelectBackward0>)
she - son token embedding:  tensor([ 0.1543,  0.4408, -0.1483, -2.3184], grad_fn=<SelectBackward0>)


In [ ]:
print("Aynı mı?")
print(torch.allclose(
    emb1[0, -1],
    emb2[0, -1]
))

Aynı mı?
True


Şu anlık ikiside aynı çünkü ikisinde de son karakter 'e' Yani token embedding tek başına bağlamı anlamıyor.

### Self-attention

Context aggregation:

Yeni bir karakter tahin ederken eski karakterleri dikkate almak istiyorsak bunu nasıl yapabiliriz?

En basit istatistiki yöntemi deneyelim. Ortalamasını almak.

In [ ]:
# Her dizinin karakter embedding'lerini ortala
context1 = emb1.mean(dim=1)
context2 = emb2.mean(dim=1)

In [ ]:
print("the context:", context1)
print("she context:", context2)

the context: tensor([[-0.4098,  0.9111, -0.8598, -1.1342]], grad_fn=<MeanBackward1>)
she context: tensor([[ 0.4852,  0.7847, -0.6573, -1.1203]], grad_fn=<MeanBackward1>)


In [ ]:
print("Aynı mı?", torch.allclose(
    context1, context2
))

Aynı mı? False


Aynı değil çünkü 'e' tokenından önceki tokenlar farklı. Ama ya kelimler 'the' ve 'hte' olsaydı ne olacaktı?

Bu iki metin aynı karakterlerden oluştuğundan aynı ortalama embedding'i üretir. Ortalama almak sıra problemini çözmez.

In [ ]:
%cd /content/drive/MyDrive/transformers

/content/drive/MyDrive/transformers
